# Frontier × cuOpt — GPU exact solve, certify, and solver-exact duals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cafzal/frontier/blob/main/examples/cuopt_colab.ipynb)

Frontier's two exact backends are co-equal: **HiGHS (CPU)** runs anywhere — including the hosted beta — while **cuOpt (GPU)** needs NVIDIA hardware, which is why it lives in this Colab template rather than the hosted deploy.

The notebook runs the full exact-audit arc — **explore with NSGA → exact GPU re-solve → certify → solver-exact duals** — on a problem you choose:

- **Path A** — load any bundled worked example by name, or
- **Path B** — scaffold **your own** problem from a fill-in template.

In normal use an AI agent drives these same four tools over MCP; here we call the tool functions in-process so everything runs inside one Colab runtime.

**Before running:** `Runtime → Change runtime type → GPU` (a T4 is plenty).

In [ ]:
# GPU sanity check — you want to see an NVIDIA device here.
!nvidia-smi -L

## Setup

Clone + editable install (the bundled `examples/` resolve relative to the repo root), then cuOpt from NVIDIA's index. The cuOpt wheel is large — expect a few minutes.

In [ ]:
!git clone --depth 1 https://github.com/cafzal/frontier.git
%cd frontier
!pip install -q -e .
!pip install -q --extra-index-url=https://pypi.nvidia.com cuopt-cu12

## Pick your problem — run **one** of the next two cells

Either path ends with a `PID` the rest of the notebook drives. (Tool responses also carry `_skill_guidance` — the workflow guidance Frontier injects for agents; we just don't print it here.)

### Path A — load a bundled worked example

In [ ]:
from mcp_server import server as frontier

# Every bundled example, by name:
print(frontier.model(action="load")["available"]["examples"])

SOURCE = "investment_portfolio"   # <-- pick any name from the list above
loaded = frontier.model(action="load", source=SOURCE)
PID = loaded["problem_id"]
loaded["status"]

### Path B — scaffold your own problem (skip if you ran Path A)

This template is a **proportional allocation with linear objectives** — the exact-LP shape, so the *entire* arc below applies, including solver-exact duals, with no extra data structures to fill in.

Exact also supports two other shapes: **binary selection** with all-`sum` objectives (a MILP — `certify` works, but integer solutions carry no duals, so `sensitivity` falls back to a frontier-inferred estimate), and **mean-variance QP** (add a minimize-direction `quadratic` objective backed by `interaction_matrices` covariance). Anything else stays with NSGA — the validate gate below will tell you.

In [ ]:
from mcp_server import server as frontier

created = frontier.model(
    action="create", name="My allocation", approach="proportional",
    objectives=[
        # 2+ objectives that genuinely conflict; direction maximize|minimize;
        # keep aggregation sum or avg for the exact-LP shape
        {"name": "Objective A", "direction": "maximize", "aggregation": "avg"},
        {"name": "Objective B", "direction": "maximize", "aggregation": "avg"},
    ],
    options=["Option 1", "Option 2", "Option 3", "Option 4"],   # 3+ options
)
PID = created["problem_id"]

# One row per (option, objective) — the matrix must be complete before solving.
# Make the scores conflict (options strong on A are weak on B) or there is no
# tradeoff to map.
frontier.model(action="update", problem_id=PID, scores=[
    {"option": "Option 1", "objective": "Objective A", "value": 8},
    {"option": "Option 1", "objective": "Objective B", "value": 3},
    {"option": "Option 2", "objective": "Objective A", "value": 6},
    {"option": "Option 2", "objective": "Objective B", "value": 5},
    {"option": "Option 3", "objective": "Objective A", "value": 4},
    {"option": "Option 3", "objective": "Objective B", "value": 7},
    {"option": "Option 4", "objective": "Objective A", "value": 2},
    {"option": "Option 4", "objective": "Objective B", "value": 9},
])["status"]

# Optional hard constraints — e.g. cap any single option's share at 40%:
# frontier.model(action="update", problem_id=PID,
#                constraints=[{"type": "max_allocation", "max": 40}])

## Gate: is cuOpt available, and does this shape qualify for exact?

`solve validate` answers both — `available` is what this environment offers, `exact_fits_shape` is whether *this problem's* formulation is exact-eligible. The gate declines unsupported shapes with a redefine hint rather than silently approximating.

In [ ]:
solvers = frontier.solve(action="validate", problem_id=PID)["solvers"]
print(solvers)
assert "cuopt" in solvers["available"], "cuOpt missing — use a GPU runtime and rerun Setup"
if not solvers["exact_fits_shape"]:
    print("NOTE: this shape is not exact-eligible —", solvers.get("exact_shape_note"))
    print("The NSGA exploration below still runs; certify/sensitivity need a supported shape.")

## 1 — Explore with the evolutionary default

NSGA is the workflow's center of gravity: fast, fits any shape, reveals the tradeoff structure. Seeded for reproducibility.

In [ ]:
nsga = frontier.solve(action="run", problem_id=PID, seed=42)
{k: nsga[k] for k in ("solver_used", "solutions_found", "frontier_complete")} | nsga["objective_ranges"]

## 2 — Exact GPU pass: `solver="cuopt"`

The exact run is stored as an **overlay** (`exact_run`) alongside the NSGA frontier — an auditor, not a replacement. Long solves run in the background; we ask for a handle immediately (`wait_seconds=0`) and poll, exactly as an agent would.

In [ ]:
import time

res = frontier.solve(action="run", problem_id=PID, solver="cuopt", seed=42, wait_seconds=0)
while res.get("status") == "running":
    print(f"  optimizing… {res['label']}  {res['elapsed_s']}s")
    time.sleep(3)
    res = frontier.solve(action="status", job_id=res["job_id"])
if res.get("error"):
    print("Exact run declined:", res["error"])  # shape gate — see the redefine hint
else:
    print({k: res.get(k) for k in ("status", "solver_used", "solutions_found")})

## 3 — Certify: audit the heuristic against the exact overlay

No params — `certify` audits the current NSGA `run` against the `exact_run` overlay. Read it as: **dominance audit** (heuristic slack the exact front catches), **coverage** (hypervolume reclaimed), the **soundness invariant** (NSGA dominates no exact point), and **corner sharpening** (the convex min-risk corner is where a QP shines and heuristics wobble).

In [ ]:
cert = frontier.explore(action="certify", problem_id=PID)
if cert.get("error"):
    print(cert["error"])
else:
    print(cert["recommendation"])
    print({
        "dominated_fraction": cert["dominance_audit"]["nsga_dominated_fraction"],
        "coverage_reclaimed": (cert["coverage"] or {}).get("reclaimed_fraction"),
        "invariant_holds": cert["invariant"]["holds"],
        "headline_corner": cert["headline_corner"],
    })

## 4 — Solver-exact duals: where to invest, what just missed

Continuous (LP/QP) exact runs carry true duals read straight from the solver — the payload names the `optimized_objective` every number is measured against. On a binary MILP, integer solutions have no duals, so this falls back to a clearly-tagged frontier-inferred estimate.

In [ ]:
sens = frontier.explore(action="sensitivity", problem_id=PID)
print("source:", sens["source"])
if sens["source"] == "solver_exact":
    print("optimized objective:", sens["optimized_objective"], "| solver:", sens["frontier_source"]["solver"])
    print("\nWhere to invest:")
    for w in sens["where_to_invest"][:3]:
        print(f"  {w['lever']:<22} {w['shadow_price']:>10.4g}   {w['interpretation']}")
    print("\nNear-misses:")
    for n in sens["near_misses"][:3]:
        print(f"  {n['option']:<22} {n['reduced_cost']:>10.4g}   {n['interpretation']}")
else:
    print(sens.get("note", ""))  # frontier-inferred fallback (MILP / heuristic run)

## 5 — See it: heuristic field vs exact overlay

Plots the first two objectives; swap `ax_x` / `ax_y` for a different pair.

In [ ]:
import matplotlib.pyplot as plt

base = frontier.explore(action="tradeoffs", problem_id=PID)["viz_data"]
exact = frontier.explore(action="tradeoffs", problem_id=PID, source="exact")
ax_x, ax_y = 0, 1
names = [o["name"] for o in base["objectives"]]

plt.figure(figsize=(7, 5))
plt.scatter([p["values"][ax_x] for p in base["points"]],
            [p["values"][ax_y] for p in base["points"]],
            s=18, alpha=0.35, label=f"NSGA ({base['provenance']['kind']})")
if "error" not in exact:
    ev = exact["viz_data"]
    plt.scatter([p["values"][ax_x] for p in ev["points"]],
                [p["values"][ax_y] for p in ev["points"]],
                s=34, marker="D", alpha=0.9, label=f"cuOpt ({ev['provenance']['kind']})")
plt.xlabel(names[ax_x]); plt.ylabel(names[ax_y])
plt.title("Heuristic field vs exact overlay")
plt.legend(); plt.show()

## Notes & troubleshooting

- **Try the other examples:** rerun Path A with any name from the printed list — `validate`'s `solvers` block tells you per-problem whether the shape qualifies for exact (the LP, QP, and binary-MILP showcases all do; quadratic-*maximize* shapes stay with NSGA by design).
- **Reproducibility:** Frontier runs cuOpt MILPs in deterministic mode automatically (`CUOPT_MIP_DETERMINISM_MODE`) — the seeded-reproducibility contract. Continuous LP/QP solves are deterministic by nature.
- **`exact_fits_shape: false`?** The gate declines what the exact math can't represent (e.g. a *maximize* quadratic, or `min`/`max` aggregations) — with a redefine hint. The NSGA frontier remains the answer for those shapes.
- **Slow first solve:** the cuOpt wheel + first GPU kernel launch dominate; subsequent solves are fast.
- **Going deeper:** `frontier.get_skill("optimization_strategy", section="Exact Solvers — Depth")` is the agent-facing guidance on when an exact pass earns its cost; the [architecture's Solver Backends section](https://github.com/cafzal/frontier/blob/main/architecture.md#solver-backends-pluggable) covers the engine design (one scalarization engine, two inner solves).
- On CPU-only machines the identical arc runs with `solver="highs"` (`pip install "frontier[highs]"`) — same certificate, same duals.